##### HIGH LEVEL IMPLEMENTATION OF THE PAPER ' Attention Is All You Need (Vaswani et al., 2017)' USING PYTORCH

here is the link to the paper: https://arxiv.org/pdf/1706.03762

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from collections import Counter, defaultdict
import torch.nn.functional as F
import re
import math
import time


In [ ]:
# amm using this for colab implemenation: uncomment this section to download the data
# import urllib.request
# import zipfile
# import os

# # Download a small, clean dataset of English-German pairs
# url = "https://www.manythings.org/anki/deu-eng.zip"
# zip_path = "deu-eng.zip"

# # Create a request with a User-Agent header to avoid HTTP Error 406
# req = urllib.request.Request(
#     url,
#     headers={
#         "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"
#     },
# )

# # Open the URL and retrieve the data
# with urllib.request.urlopen(req) as response:
#     with open(zip_path, "wb") as out_file:
#         out_file.write(response.read())

# # Extract it
# with zipfile.ZipFile(zip_path, "r") as zip_ref:
#     zip_ref.extractall(".")

In [ ]:
with open("../deu.txt", "r", encoding="utf-8") as f:
    lines = f.read().strip().split("\n")

# # for google colab
# with open("deu.txt", "r", encoding="utf-8") as f:
#     lines = f.read().strip().split("\n")


pairs = []

eng_sentences = []
german_sentences = []


for line in lines[:50000]:
    parts = line.split("\t")

    if len(parts) < 2:
        continue

    english = parts[0]
    german = parts[1]

    pairs.append((english, german))

    eng_sentences.append(english)
    german_sentences.append(german)


print(f"Number of sentence pairs: {len(pairs)}")

print(f"Sample pair: {pairs[:3]}")

In [ ]:
class VocabularyBuilder:
    def __init__(self):
        # Special tokens required for translation models
        self.word2idx = {
            "<PAD>": 0,
            "<UNK>": 1,
            "<SOS>": 2,
            "<EOS>": 3,
            "</w>": 4,
        }
        self.idx2word = {v: k for k, v in self.word2idx.items()}
        self.merge_order = []
        self.num_words = len(self.word2idx)

        # This will hold our word frequency dictionary
        self.word_freqs = Counter()

    def preprocess_and_count(self, all_sentences):
        """
        Processes sentences and builds a word frequency dictionary.
        This runs once and saves massive amounts of memory and time.
        """
        raw_word_counts = Counter()

        for sentence in all_sentences:
            sentence = sentence.lower()
            # Separate punctuation
            sentence = re.sub(r'([.,!?;:"\'()])', r" \1 ", sentence)
            words = sentence.strip().split()
            raw_word_counts.update(words)

        # Convert words into spaces-separated characters with an end-of-word symbol
        self.word_freqs = Counter()
        for word, count in raw_word_counts.items():
            # Example: 'hug' -> 'h u g </w>'
            char_tuple = tuple(list(word) + ["</w>"])
            self.word_freqs[char_tuple] = count
        # Initialize basic single-character vocabulary
        self.build_initial_vocabulary()

    def build_initial_vocabulary(self):
        """Extracts unique base characters from our counted words."""
        for word_tuple in self.word_freqs.keys():
            for char in word_tuple:
                if char not in self.word2idx:
                    self.word2idx[char] = self.num_words
                    self.idx2word[self.num_words] = char
                    self.num_words += 1

    def get_pair_counts(self):
        """
        Fast pair counting using the pre-calculated word frequencies.
        """
        pairs = defaultdict(int)
        for word_tuple, freq in self.word_freqs.items():
            for i in range(len(word_tuple) - 1):
                pair = (word_tuple[i], word_tuple[i + 1])
                pairs[pair] += freq
        # print("pairs: ", pairs)
        return pairs

    def merge_vocab(self, pair):
        """
        Merges the most frequent pair in-place across the vocabulary.
        """
        new_word_freqs = Counter()
        bigram = re.escape(" ".join(pair))
        # Look for the pair with a boundary or space around it
        p = re.compile(r"(?<!\S)" + bigram + r"(?!\S)")
        # Save the merge order rule
        merged_string = "".join(pair)
        self.merge_order.append(pair)

        # Add the new merged token to the vocabulary if it isn't there
        if merged_string not in self.word2idx:
            self.word2idx[merged_string] = self.num_words
            self.idx2word[self.num_words] = merged_string
            self.num_words += 1

        # Update the character tuple configurations
        for word_tuple, freq in self.word_freqs.items():
            word_str = " ".join(word_tuple)
            new_word_str = p.sub(merged_string, word_str)
            new_word_tuple = tuple(new_word_str.split())
            new_word_freqs[new_word_tuple] = freq

        self.word_freqs = new_word_freqs

    def fit(self, all_sentences, num_merges):
        """
        Main function to run the BPE algorithm.
        """
        self.preprocess_and_count(all_sentences)

        for i in range(num_merges):
            pairs = self.get_pair_counts()
            if not pairs:
                break
            # Find the most common pair
            best_pair = max(pairs, key=pairs.get)
            self.merge_vocab(best_pair)
            # Optional progress information
            if (i + 1) % 500 == 0:
                print(f"Completed {i + 1} merges")

        print(f"Final vocabulary size: {self.num_words}")

        print(f"Total merges learned: {len(self.merge_order)}")

    def _word_tokenizer(self, word):
        word_tuple = tuple(list(word) + ["</w>"])

        for pair in self.merge_order:
            new_word = []
            i = 0
            while i < len(word_tuple):
                if (
                    i < len(word_tuple) - 1
                    and word_tuple[i] == pair[0]
                    and word_tuple[i + 1] == pair[1]
                ):
                    # when get match, Fuse them together
                    new_word.append(pair[0] + pair[1])
                    i += 2  # Skip the next character since we consumed it
                else:
                    new_word.append(word_tuple[i])
                    i += 1
            word_tuple = tuple(new_word)

        return word_tuple

    def numericalize(self, sentence):
        sentence = sentence.lower()
        sentence = re.sub(r'([.,!?;:"\'()])', r" \1 ", sentence)
        words = sentence.strip().split()

        tokens = []
        # 3. Process every word through the BPE tokenizer
        for word in words:
            bpe_tokens = self._word_tokenizer(word)
            tokens.extend(bpe_tokens)

        numerical_ids = [
            self.word2idx.get(token, self.word2idx["<UNK>"]) for token in tokens
        ]

        return numerical_ids


In [ ]:
tokenizer = VocabularyBuilder()

all_sentences = eng_sentences + german_sentences

tokenizer.fit(all_sentences, num_merges=5000)

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, pairs, tokenizer):

        self.pairs = pairs
        self.tokenizer = tokenizer

    def __len__(self):

        return len(self.pairs)

    def __getitem__(self, idx):

        src_sent, trg_sent = self.pairs[idx]

        src_tokens = self.tokenizer.numericalize(src_sent)

        trg_tokens = self.tokenizer.numericalize(trg_sent)

        src_tensor = torch.tensor(
            src_tokens + [self.tokenizer.word2idx["<EOS>"]], dtype=torch.long
        )

        trg_tensor = torch.tensor(
            [self.tokenizer.word2idx["<SOS>"]]
            + trg_tokens
            + [self.tokenizer.word2idx["<EOS>"]],
            dtype=torch.long,
        )

        return (src_tensor, trg_tensor)


In [ ]:
def collate_fn(batch):

    src_batch, trg_batch = zip(*batch)

    src_batch = pad_sequence(src_batch, padding_value=0, batch_first=True)

    trg_batch = pad_sequence(trg_batch, padding_value=0, batch_first=True)

    return (src_batch, trg_batch)


In [ ]:
dataset = TranslationDataset(pairs, tokenizer)


dataloader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)


src_batch, trg_batch = next(iter(dataloader))

print("\nSource batch shape:", src_batch.shape)

print("Target batch shape:", trg_batch.shape)

print("\nSource batch:")

print(src_batch)

print("\nTarget batch:")

print(trg_batch)

In [ ]:
class EmbeddingLayer(nn.Module):
    def __init__(self, vocab_size, d_model, max_seq_len=5000):
        super().__init__()
        self.vocab_size = vocab_size
        self.emb_dim = d_model
        self.max_seq_len = max_seq_len
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.dropout = nn.Dropout(0.1)
        pos_enc = self._calculatePositionalEncoding3()
        self.register_buffer("pos_enc", pos_enc)

    # src is 2D
    def _calculatePositionalEncoding1(self):
        if self.emb_dim % 2 != 0:
            raise ValueError("emb_dim must be even for positional encoding")

        pos_enc = torch.empty(self.max_seq_len, self.emb_dim)
        for pos in range(self.max_seq_len):
            dim = 0
            while dim < self.emb_dim // 2:
                dim_param = (2 * dim) / self.emb_dim

                sinusoid_param = pos / math.pow(10000, dim_param)
                sin_pos_enc = math.sin(sinusoid_param)
                pos_enc[pos, 2 * dim] = sin_pos_enc
                cos_pos_enc = math.cos(sinusoid_param)
                pos_enc[pos, 2 * dim + 1] = cos_pos_enc
                dim += 1

        # positional_tensor = pos_enc.unsqueeze(0)

        return pos_enc

    def _calculatePositionalEncoding2(self):
        if self.emb_dim % 2 != 0:
            raise ValueError("emb_dim must be even for positional encoding")

        pos = torch.arange(self.max_seq_len, dtype=torch.float32)

        dim_indices = torch.arange(0, self.emb_dim, 2, dtype=torch.float32)
        denom = torch.pow(10000.0, dim_indices / self.emb_dim)

        angles = pos.unsqueeze(1) / denom.unsqueeze(0)

        pos_enc = torch.zeros(self.max_seq_len, self.emb_dim)

        pos_enc[:, 0::2] = torch.sin(angles)
        pos_enc[:, 1::2] = torch.cos(angles)

        # positional_tensor = pos_enc.unsqueeze(0)

        return pos_enc

    def _calculatePositionalEncoding3(self):
        if self.emb_dim % 2 != 0:
            raise ValueError("emb_dim must be even for positional encoding")
        pos = torch.arange(self.max_seq_len, dtype=torch.float32)

        dim_indices = torch.arange(0, self.emb_dim, 2, dtype=torch.float32)
        inverted_frequency = torch.exp(
            (-1 * dim_indices) * (math.log(10000) / self.emb_dim)
        )
        angles = pos.unsqueeze(1) * inverted_frequency.unsqueeze(0)

        pos_enc = torch.zeros(self.max_seq_len, self.emb_dim)

        pos_enc[:, 0::2] = torch.sin(angles)
        pos_enc[:, 1::2] = torch.cos(angles)

        # positional_tensor = pos_enc.unsqueeze(0)

        return pos_enc

    def forward(self, src):
        batch_size, seq_len = src.shape
        if seq_len > self.max_seq_len:
            raise ValueError(
                f"Sequence length ({seq_len}) exceeds max_seq_len ({self.max_seq_len})"
            )

        embedded = self.embedding(src) * math.sqrt(self.emb_dim)
        pos_enc = self.pos_enc.unsqueeze(0)
        updated_embedded = embedded + pos_enc[:, :seq_len, :]

        output_embedding = self.dropout(updated_embedded)

        return output_embedding

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, h, d_model, d_k, d_v, max_seq_len=5000):
        super().__init__()
        self.h = h
        self.d_k = d_k
        self.d_v = d_v
        self.Wq = nn.Linear(d_model, h * d_k)
        self.Wk = nn.Linear(d_model, h * d_k)
        self.Wv = nn.Linear(d_model, h * d_v)
        self.feedforward = nn.Linear(h * d_v, d_model)

        self.register_buffer(
            "causal_mask",
            torch.triu(torch.ones(max_seq_len, max_seq_len), diagonal=1).bool(),
        )

    def forward(
        self, query_source, key_source, value_source, mask=False, padd_mask=None
    ):
        batch_size = query_source.size(0)
        seq_len = query_source.size(1)

        # Project and reshape
        Q = (
            self.Wq(query_source)
            .view(batch_size, seq_len, self.h, self.d_k)
            .transpose(1, 2)
        )
        K = self.Wk(key_source).view(batch_size, -1, self.h, self.d_k).transpose(1, 2)
        V = self.Wv(value_source).view(batch_size, -1, self.h, self.d_v).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # Apply masks
        if mask:
            scores.masked_fill_(self.causal_mask[:seq_len, :seq_len], float("-inf"))

        if padd_mask is not None:
            scores.masked_fill_(padd_mask.unsqueeze(1).unsqueeze(2), float("-inf"))

        # Attention weights and output
        attention = F.softmax(scores, dim=-1)
        output = torch.matmul(attention, V)

        # Combine heads and project
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        return self.feedforward(output)

In [ ]:
class AttentionSublayer(nn.Module):
    def __init__(self, h, d_model, d_k, d_v, dropout=0.1):
        super().__init__()
        self.multi_head_attention_sublayer = MultiHeadAttention(h, d_model, d_k, d_v)

        self.normalization = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self, src, query_source, key_source, value_source, mask=False, padd_mask=None
    ):
        attention_output = self.multi_head_attention_sublayer(
            query_source=query_source,
            key_source=key_source,
            value_source=value_source,
            mask=mask,
            padd_mask=padd_mask,
        )

        add_dropout = self.dropout(attention_output)
        add_residual = src + add_dropout
        output = self.normalization(add_residual)

        return output


In [ ]:
class FeedForwardSublayer(nn.Module):
    def __init__(self, d_model, d_ff=None, dropout=0.1):
        super().__init__()
        if d_ff is None:
            d_ff = 4 * d_model
        self.feedforward1 = nn.Linear(d_model, d_ff)
        self.feedforward2 = nn.Linear(d_ff, d_model)
        self.normalization = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, src):
        forward1 = self.feedforward1(src)
        activated1 = self.relu(forward1)
        forward2 = self.feedforward2(activated1)

        add_dropout = self.dropout(forward2)
        add_residual = src + add_dropout
        output = self.normalization(add_residual)

        return output


In [ ]:
class Encoder(nn.Module):
    def __init__(self, h, d_model, d_k, d_v, embedding, n_layers=6):
        super().__init__()
        self.embedding_layer = embedding
        self.layers = nn.ModuleList(
            [
                nn.ModuleDict(
                    {
                        "attention": AttentionSublayer(
                            h=h, d_model=d_model, d_k=d_k, d_v=d_v
                        ),
                        "feed_forward": FeedForwardSublayer(d_model=d_model),
                    }
                )
                for layer in range(n_layers)
            ]
        )

    def forward(self, src):
        padd_mask = src == 0

        encoded = self.embedding_layer(src)
        for layer in self.layers:
            attention = layer["attention"](
                src=encoded,
                query_source=encoded,
                key_source=encoded,
                value_source=encoded,
                padd_mask=padd_mask,
            )
            feedforward_output = layer["feed_forward"](attention)
            encoded = feedforward_output

        return encoded

In [ ]:
class Decoder(nn.Module):
    def __init__(self, h, d_model, d_k, d_v, embedding, n_layers=6):
        super().__init__()
        self.embedding_layer = embedding
        self.layers = nn.ModuleList(
            [
                nn.ModuleDict(
                    {
                        "self_attention": AttentionSublayer(h, d_model, d_k, d_v),
                        "cross_attention": AttentionSublayer(h, d_model, d_k, d_v),
                        "feed_forward": FeedForwardSublayer(d_model),
                    }
                )
                for layers in range(n_layers)
            ]
        )

    def forward(self, src, encoder_output, src_padd_mask=None):
        trg_padd_mask = src == 0

        decoded = self.embedding_layer(src)
        for layer in self.layers:
            self_attention = layer["self_attention"](
                src=decoded,
                query_source=decoded,
                key_source=decoded,
                value_source=decoded,
                mask=True,
                padd_mask=trg_padd_mask,
            )
            cross_attention = layer["cross_attention"](
                src=self_attention,
                query_source=self_attention,
                key_source=encoder_output,
                value_source=encoder_output,
                mask=False,
                padd_mask=src_padd_mask,
            )
            feedforward_output = layer["feed_forward"](cross_attention)
            decoded = feedforward_output

        output = torch.matmul(decoded, self.embedding_layer.embedding.weight.T)
        return output


In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(
        self,
        src,
        trg,
        # vocab_size,
    ):
        src_padd_mask = src == 0

        # batch_size = src.shape[0]
        # trg_len = trg.shape[1]

        # outputs = torch.zeros(batch_size, trg_len, vocab_size).to(self.device)
        encoder_output = self.encoder(src)

        decoder_output = self.decoder(trg, encoder_output, src_padd_mask)

        return decoder_output

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VOCAB_SIZE = tokenizer.num_words
D_MODEL = 512
D_V = 64
D_K = 64
H = 8

embedding_layer = EmbeddingLayer(
    vocab_size=VOCAB_SIZE, d_model=D_MODEL, max_seq_len=5000
)
enc = Encoder(h=H, d_model=D_MODEL, d_k=D_K, d_v=D_V, embedding=embedding_layer)
dec = Decoder(h=H, d_model=D_MODEL, d_k=D_K, d_v=D_V, embedding=embedding_layer)
model = Seq2Seq(encoder=enc, decoder=dec, device=device).to(device)

beta_1 = 0.9
beta_2 = 0.98
epsilon = 1e-9
label_smoothing = 0.1
N_EPOCHS = 10

total_steps = N_EPOCHS * len(dataloader)

warmup_steps = 0.4 * total_steps


optimizer = torch.optim.Adam(
    model.parameters(),
    betas=(beta_1, beta_2),
    eps=epsilon,
    lr=1.0,
)


def getLearningRate(step):
    step_num = step + 1
    lrate = (D_MODEL ** (-0.5)) * min(
        (step_num ** (-0.5)), (step_num * (warmup_steps ** (-1.5)))
    )
    return lrate


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=getLearningRate)

criterion = nn.CrossEntropyLoss(
    ignore_index=0,
    label_smoothing=label_smoothing,
)  # Ignore <PAD> tokens in loss calculation

print(f"Model initialized successfully. Device: {device}")

In [ ]:
def train(model, dataloader, optimizer, criterion):

    model.train()
    epoch_loss = 0

    for i, (src, trg) in enumerate(dataloader):
        src = src.to(device)
        trg = trg.to(device)

        optimizer.zero_grad()
        output = model(src, trg[:, :-1])

        flat_trg = trg[:, 1:].reshape(-1)
        flat_output = output.reshape(-1, VOCAB_SIZE)

        loss = criterion(flat_output, flat_trg)
        loss.backward()

        optimizer.step()
        scheduler.step()
        epoch_loss += loss.item()

    return epoch_loss / len(dataloader)


for epoch in range(N_EPOCHS):
    start_time = time.time()

    train_loss = train(model, dataloader, optimizer, criterion)
    end_time = time.time()

    print(f"Epoch: {epoch + 1:02} | Time: {end_time - start_time:.2f}s")
    print(
        f"\tTrain Loss: {train_loss:.3f} | Jarring Perplexity: {math.exp(train_loss):.3f}"
    )

In [ ]:
def translate_sentence(sentence, device, tokenizer, model, max_len=500):
    model.eval()
    tokens = tokenizer.numericalize(sentence)
    tokens = tokens + [tokenizer.word2idx["<EOS>"]]
    src_tensor = torch.LongTensor(tokens).unsqueeze(0).to(device)
    with torch.no_grad():
        encoder_output = model.encoder(src_tensor)

    trg_indexes = [tokenizer.word2idx["<SOS>"]]
    src_padd_mask = src_tensor == 0

    for _ in range(max_len):
        trg_tensor = torch.LongTensor(trg_indexes).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = model.decoder(trg_tensor, encoder_output, src_padd_mask)

        pred_token = logits[-1, -1, :].argmax().item()
        trg_indexes.append(pred_token)
        # Stop if the model predicts the End of Sentence token
        if pred_token == tokenizer.word2idx["<EOS>"]:
            break
    # 6. Convert integer indices back into actual German words
    trg_tokens = [tokenizer.idx2word[i] for i in trg_indexes]

    # Return string while cutting out the structural control symbols
    return " ".join(trg_tokens[1:-1])


example_input = "hello, how are you"
translation = translate_sentence(example_input, device, tokenizer, model)
print(f"English: {example_input}")
print(f"German Translation: {translation}")
